# Generate dataset with GW data

Imports, include the project root in the Python path.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal as _lal

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

%reload_ext autoreload
%autoreload 2

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())
print("sys.path:", sys.path[:5])

PROJECT_ROOT: /home/victor/gw/cbc_pe
cwd: /home/victor/gw/cbc_pe/notebooks
sys.path: ['/home/victor/gw/cbc_pe', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '']


In [2]:
from src.config import SimulationConfig
from src.parameters import CBCParameters
from src.sampling import ParameterSampler
from src.waveform import WaveformGenerator
from src.detectors import DetectorProjector
from src.noise import NoiseModel
from src.injection import SignalInjector
from src.processing import SignalProcessor
from src.labels import LabelTransformer
from src.dataset import DatasetBuilder

#### Para sistemas cuya duración en banda excede la ventana de análisis, el dataset no contiene la inspiral completa. Se conserva la porción final de la señal dentro de una ventana fija de 4 s, garantizando que al menos el último segundo antes de la coalescencia esté presente. El SNR y el reescalado de distancia se calculan sobre la señal efectivamente usada para la inyección.

In [ ]:
config = SimulationConfig(
    duration=4.0,
    processing_context_start_samples=1664, # This is the equivalent of  1664 / 4096 = 0.40625 seconds margin used in the processing 
    processing_context_end_samples=1664,
)

noise_model = NoiseModel(config)

print(config.length)
print(config.processing_length)
print(config.duration)
print(config.processing_duration)

psd_out = noise_model.get_psd("H1")
psd_proc = noise_model.get_psd("H1", length=config.processing_length)

print(len(psd_out), config.flength, psd_out.delta_f, config.delta_f)
print(len(psd_proc), config.processing_flength, psd_proc.delta_f, config.processing_delta_f)

noise_out = noise_model.sample("H1")
noise_proc = noise_model.sample("H1", length=config.processing_length)

print(len(noise_out), config.length)
print(len(noise_proc), config.processing_length)

16384
19712
4.0
4.8125
8193 8193 0.25 0.25
9857 9857 0.2077922077922078 0.2077922077922078
16384 16384
19712 19712


In [4]:
config = SimulationConfig(
    duration=4.0,
    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

noise_model = NoiseModel(config)

processor = SignalProcessor(
    config=config,
    whitening_method="psd",
    apply_highpass=True,
    apply_lowpass=True,
    preserve_length=False,
    output_mode="crop_to_config",
    highpass_frequency=30.0,
    lowpass_frequency=512.0,
    fir_order=256,
    fir_beta=5.0,
    remove_corrupted=True,
)

noise = noise_model.sample("H1", length=config.processing_length)
noise.start_time = 1000.0

psd = noise_model.get_psd("H1", length=config.processing_length)

processed = processor.process(
    strain=noise,
    psd=psd,
    detector_name="H1",
)

print(len(noise), config.processing_length)
print(len(processed), config.length)
print(float(noise.start_time))
print(float(processed.start_time))
print(float(noise.start_time) + config.processing_context_start_seconds)

19712 19712
16384 16384
1000.0
1000.40625
1000.40625


In [5]:

config = SimulationConfig(
    duration=4.0,
    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

injector = SignalInjector(config)

zero_out = injector.build_zero_strain(start_time=1000.0)
zero_ctx = injector.build_zero_strain(
    start_time=1000.0,
    length=config.processing_length,
)

print(len(zero_out), config.length)
print(len(zero_ctx), config.processing_length)

zero_ctx2 = injector.set_strain_start_time(
    zero_ctx,
    start_time=999.59375,
    expected_length=config.processing_length,
)

print(len(zero_ctx2), float(zero_ctx2.start_time))

16384 16384
19712 19712
19712 999.59375


In [6]:
signal = injector.build_zero_strain(
    start_time=1000.5,
    length=1000,
)

signal[:] = 1.0

context = injector.build_zero_strain(
    start_time=1000.0,
    length=config.processing_length,
)

result = injector.inject(context, signal)

print(result.is_partially_clipped)
print(result.n_signal_samples, result.n_injected_samples)
print(result.signal_start_index, result.signal_end_index)
print(len(result.strain), config.processing_length)

False
1000 1000
2048 3048
19712 19712


In [8]:
import numpy as np


# Ajusta estos kwargs a los que estés usando ahora
signal_processor_kwargs = {
    "whitening_method": "psd",
    "apply_highpass": True,
    "apply_lowpass": True,
    "apply_standardization": False,
    "output_mode": "crop_to_config",

    "whitening_low_frequency_cutoff": 30.0,
    "whitening_max_filter_duration": 0.5,
    "whitening_trunc_method": "hann",

    "highpass_frequency": 30.0,
    "lowpass_frequency": 512.0,

    "fir_order": 256,
    "fir_beta": 5.0,
    "remove_corrupted": True,
}

config = SimulationConfig(
    duration=4.0,
    safe_margin_start=0.0,
    safe_margin_end=0.0,
    processing_context_start_samples=1664,
    processing_context_end_samples=1664,
)

rng = np.random.default_rng(1234)

builder = DatasetBuilder.from_config(
    config=config,
    signal_processor_kwargs=signal_processor_kwargs,
    rng=rng,
)

# 1. Muestreamos parámetros y construimos waveform/proyección manualmente
params = builder.parameter_sampler.sample_one()
geocentric_coalescence_time = builder._sample_geocentric_coalescence_time()

waveform = builder.waveform_generator.generate(params)

projection = builder.detector_projector.project(
    h_plus=waveform.h_plus,
    h_cross=waveform.h_cross,
    parameters=params,
    geocentric_coalescence_time=geocentric_coalescence_time,
)

print("Projected detectors:", projection.strains.keys())

# 2. Check con max_duration = 3.2 s
windowed_32 = builder.network_window_selector.select(
    projected_strains=projection.strains,
    max_duration=3.2,
)

print("\n--- max_duration = 3.2 ---")
print("used_window_duration:", windowed_32.metadata.used_window_duration)
print("max_window_duration:", windowed_32.metadata.max_window_duration)
print("is_truncated:", windowed_32.metadata.is_truncated)
print("fraction_used:", windowed_32.metadata.fraction_network_duration_used)

assert windowed_32.metadata.used_window_duration <= 3.2 + config.delta_t
assert abs(windowed_32.metadata.max_window_duration - 3.2) < 1e-9

# 3. Check con max_duration = 4.0 s
windowed_40 = builder.network_window_selector.select(
    projected_strains=projection.strains,
    max_duration=4.0,
)

print("\n--- max_duration = 4.0 ---")
print("used_window_duration:", windowed_40.metadata.used_window_duration)
print("max_window_duration:", windowed_40.metadata.max_window_duration)
print("is_truncated:", windowed_40.metadata.is_truncated)
print("fraction_used:", windowed_40.metadata.fraction_network_duration_used)

assert windowed_40.metadata.used_window_duration <= 4.0 + config.delta_t
assert abs(windowed_40.metadata.max_window_duration - 4.0) < 1e-9

print("\nWindowing checks passed.")

Projected detectors: dict_keys(['H1', 'L1', 'V1'])

--- max_duration = 3.2 ---
used_window_duration: 0.305908203125
max_window_duration: 3.2
is_truncated: False
fraction_used: 1.0

--- max_duration = 4.0 ---
used_window_duration: 0.305908203125
max_window_duration: 4.0
is_truncated: False
fraction_used: 1.0

Windowing checks passed.


In [9]:
batch = builder.build_dataset(num_samples=10, progress_every=1)

X = batch.X
metadata = batch.metadata

assert X.shape == (10, 3, config.length)

for i, m in enumerate(metadata):
    ctx = m["processing_context"]
    placement = m["placement"]

    assert abs(ctx["output_segment_start_time"] - placement["segment_start_time"]) < 1e-9
    assert abs(ctx["output_segment_end_time"] - placement["segment_end_time"]) < 1e-9

    for det in m["detectors"]:
        inj = m["injection"][det]

        assert abs(inj["segment_start_time"] - ctx["context_segment_start_time"]) < config.delta_t
        assert abs(inj["segment_end_time"] - ctx["context_segment_end_time"]) < config.delta_t

        assert inj["is_partially_clipped"] is False
        assert inj["n_signal_samples"] == inj["n_injected_samples"]

    assert m["windowing"]["used_window_duration"] <= config.duration + config.delta_t

print("Context dataset checks passed.")

Built sample 1 of 10 --> 10.0% completed (attempts=1, failed=0, elapsed=1.1s, 1.12s/sample, 0.89 samples/s)
Built sample 2 of 10 --> 20.0% completed (attempts=2, failed=0, elapsed=2.2s, 1.08s/sample, 0.93 samples/s)
Built sample 3 of 10 --> 30.0% completed (attempts=3, failed=0, elapsed=2.9s, 0.96s/sample, 1.04 samples/s)
Built sample 4 of 10 --> 40.0% completed (attempts=4, failed=0, elapsed=3.6s, 0.90s/sample, 1.11 samples/s)
Built sample 5 of 10 --> 50.0% completed (attempts=5, failed=0, elapsed=4.4s, 0.87s/sample, 1.15 samples/s)
Built sample 6 of 10 --> 60.0% completed (attempts=6, failed=0, elapsed=5.4s, 0.90s/sample, 1.11 samples/s)
Built sample 7 of 10 --> 70.0% completed (attempts=7, failed=0, elapsed=6.4s, 0.92s/sample, 1.09 samples/s)
Built sample 8 of 10 --> 80.0% completed (attempts=8, failed=0, elapsed=7.2s, 0.90s/sample, 1.12 samples/s)
Built sample 9 of 10 --> 90.0% completed (attempts=9, failed=0, elapsed=8.2s, 0.91s/sample, 1.10 samples/s)
Built sample 10 of 10 --> 10

In [10]:
snrs = np.array([
    m["snr"]["final_network_snr"]
    for m in metadata
])

assert np.all(snrs >= 10.0 - 1e-4)
assert np.all(snrs <= 25.0 + 1e-4)